# **Medición del nivel de apropiación TIC en la comunidad de Puerto Gaitan**
#### Despliegue - Trabajo final - Karen Mejía Sánchez, ID 000581891

#### Aprendizaje de máquina - Maestría en ciencia de datos

In [ ]:
import pandas as pd

simulated_new_data = pd.read_csv('/content/DatosFuturos_Medicion_del_nivel_de_apropiacion_TIC.csv', sep=';', encoding='latin1')
display(simulated_new_data.head())

,Nombre,Edad,Genero,¿Con qué dispositivos tecnológicos te conectas regularmente?,¿Mayormente utilizas internet en tu vida diaria para,¿Has recibido formación formal sobre el uso de Tecnologías de la Información y Comunicaciones?,¿Cuáles son las principales barreras que enfrentas para utilizar las Tecnologías de la Información y Comunicaciones?,¿Te gustaría recibir formación sobre las Tecnologías de la Información y Comunicaciones?,¿Con qué frecuencia accedes a internet?,¿Qué tan importante consideras el uso de Tecnologías de la Información y Comunicaciones en tu vida diaria?,¿Cuál es tu nivel educativo?,¿A que grupo poblacional pertences?
0,Juan Lopez,18-24,Masculino,Computador,Estudio,Talleres o cursos,Falta de acceso,Si,Diariamente,Muy importante,Pregrado,Ninguno
1,Susana Alvarez,25-34,Femenino,Computador,Estudio,Ninguno,Ninguna,No,Diariamente,Muy importante,Técnica o tecnólogo,Ninguno
2,Andres Perez,18-24,Masculino,Computador,Trabajo,Ninguno,Ninguna,No,Diariamente,Muy importante,Secundaria,Indígena
3,Mariana Montoya Rios,0 -18,Femenino,Celular,Redes sociales,Autoaprendizaje,Falta de habilidades,No,Diariamente,Importante,Secundaria,Ninguno
4,Sandra Rojas,45-54,Femenino,Celular,Redes sociales,Ninguno,Ninguna,No,Escasamente,Muy importante,Técnica o tecnólogo,Afrocolombiana


In [ ]:
import joblib
import unicodedata

def clean_special_characters(df):
    """
    Limpia caracteres especiales (como tildes), reemplaza espacios por guiones bajos y elimina signos de interrogación
    de todas las columnas de tipo string y de los nombres de las columnas en un DataFrame.
    """
    df_cleaned = df.copy()

    # Limpiar nombres de columnas
    new_columns = []
    for col in df_cleaned.columns:
        cleaned_col = unicodedata.normalize('NFKD', str(col)).encode('ascii', 'ignore').decode('utf-8')
        cleaned_col = cleaned_col.replace(' ', '_')  # Reemplazar espacios por guiones bajos
        cleaned_col = cleaned_col.replace('?', '')   # Eliminar signos de interrogación
        new_columns.append(cleaned_col)
    df_cleaned.columns = new_columns

    # Limpiar valores en columnas de tipo string
    for col in df_cleaned.select_dtypes(include=['object']).columns:
        df_cleaned[col] = df_cleaned[col].apply(lambda x: unicodedata.normalize('NFKD', str(x)).encode('ascii', 'ignore').decode('utf-8') if pd.notna(x) else x)
    return df_cleaned

def preprocess_new_data(new_raw_df, ohe_path='one_hot_encoder.joblib', le_path='label_encoder.joblib'):
    # Cargar encoders
    ohe = joblib.load(ohe_path)
    le = joblib.load(le_path)

    # 1. Limpieza de nombres de columnas y valores
    cleaned_df = clean_special_characters(new_raw_df.copy())

    processed_df = cleaned_df.copy() # Usar la copia limpia como punto de partida

    # 2. Eliminar columnas que no deben ser consideradas por el modelo
    columns_to_drop_for_prediction = ['Nombre', 'Genero', 'Edad', 'A_que_grupo_poblacional_pertences']
    processed_df = processed_df.drop(columns=columns_to_drop_for_prediction, errors='ignore')

    # 3. Definir características categóricas para OHE
    # Las columnas restantes después de eliminar las especificadas son las características categóricas.
    categorical_features_for_ohe = [col for col in processed_df.columns]

    # 4. Asegurarse de que las columnas identificadas sean de tipo string para el OHE
    for col in categorical_features_for_ohe:
        processed_df[col] = processed_df[col].astype('category')

    # 5. Transformar con OneHotEncoder cargado
    encoded_data = ohe.transform(processed_df[categorical_features_for_ohe])

    # 6. Crear DataFrame con nombres de columnas correctos (los que el modelo espera)
    encoded_df = pd.DataFrame(encoded_data, columns=ohe.get_feature_names_out(categorical_features_for_ohe))
    display(encoded_df)

    return encoded_df, le # Retornamos el le por si se necesita para decodificar

def predict_new_data(new_raw_df, model_path='best_logistic_regression_model.joblib'):
    model = joblib.load(model_path)
    processed_features, label_encoder = preprocess_new_data(new_raw_df)
    predictions_numeric = model.predict(processed_features)
    predictions_decoded = label_encoder.inverse_transform(predictions_numeric)
    return predictions_decoded

# Realizar predicción
final_prediction = predict_new_data(simulated_new_data)
print(f"\nLa predicción para los datos simulados es: {final_prediction}")

,Con_que_dispositivos_tecnologicos_te_conectas_regularmente_Celular,Con_que_dispositivos_tecnologicos_te_conectas_regularmente_Computador,Con_que_dispositivos_tecnologicos_te_conectas_regularmente_Tablet,Mayormente_utilizas_internet_en_tu_vida_diaria_para_Compras en linea,Mayormente_utilizas_internet_en_tu_vida_diaria_para_Estudio,Mayormente_utilizas_internet_en_tu_vida_diaria_para_Plataformas de streaming,Mayormente_utilizas_internet_en_tu_vida_diaria_para_Redes sociales,Mayormente_utilizas_internet_en_tu_vida_diaria_para_Trabajo,Has_recibido_formacion_formal_sobre_el_uso_de_Tecnologias_de_la_Informacion_y_Comunicaciones_Autoaprendizaje,Has_recibido_formacion_formal_sobre_el_uso_de_Tecnologias_de_la_Informacion_y_Comunicaciones_Ninguno,...,Con_que_frecuencia_accedes_a_internet_Semanalmente,Que_tan_importante_consideras_el_uso_de_Tecnologias_de_la_Informacion_y_Comunicaciones_en_tu_vida_diaria_Importante,Que_tan_importante_consideras_el_uso_de_Tecnologias_de_la_Informacion_y_Comunicaciones_en_tu_vida_diaria_Muy importante,Que_tan_importante_consideras_el_uso_de_Tecnologias_de_la_Informacion_y_Comunicaciones_en_tu_vida_diaria_Poco importante,Cual_es_tu_nivel_educativo_Postgrado,Cual_es_tu_nivel_educativo_Pregrado,Cual_es_tu_nivel_educativo_Primaria,Cual_es_tu_nivel_educativo_Secundaria,Cual_es_tu_nivel_educativo_Sin educacion formal,Cual_es_tu_nivel_educativo_Tecnica o tecnologo
0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
5,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
7,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0



La predicción para los datos simulados es: ['Muy alta' 'Muy alta' 'Muy alta' 'Media' 'Media' 'Muy alta' 'Alta'
 'Media']


In [ ]:
import pandas as pd

# Asegurarse de que final_prediction sea una Serie o lista para concatenar
# Si final_prediction es un array de numpy, convertirlo a una Serie de pandas
if isinstance(final_prediction, (list, pd.Series)):
    predictions_series = pd.Series(final_prediction, name='Nivel_TIC_Predicho')
elif hasattr(final_prediction, 'tolist'): # Check if it's a numpy array or similar
    predictions_series = pd.Series(final_prediction.tolist(), name='Nivel_TIC_Predicho')
else:
    predictions_series = pd.Series(final_prediction, name='Nivel_TIC_Predicho') # Fallback

# Asegurarse de que la longitud de ambos coincida antes de concatenar
if len(simulated_new_data) == len(predictions_series):
    results_df = pd.concat([simulated_new_data, predictions_series], axis=1)
    display(results_df)
else:
    print(f"Error: La longitud de los datos de entrada ({len(simulated_new_data)}) no coincide con la longitud de las predicciones ({len(predictions_series)}).")
    results_df = None # Set to None to indicate an error or mismatch


,Nombre,Edad,Genero,¿Con qué dispositivos tecnológicos te conectas regularmente?,¿Mayormente utilizas internet en tu vida diaria para,¿Has recibido formación formal sobre el uso de Tecnologías de la Información y Comunicaciones?,¿Cuáles son las principales barreras que enfrentas para utilizar las Tecnologías de la Información y Comunicaciones?,¿Te gustaría recibir formación sobre las Tecnologías de la Información y Comunicaciones?,¿Con qué frecuencia accedes a internet?,¿Qué tan importante consideras el uso de Tecnologías de la Información y Comunicaciones en tu vida diaria?,¿Cuál es tu nivel educativo?,¿A que grupo poblacional pertences?,Nivel_TIC_Predicho
0,Juan Lopez,18-24,Masculino,Computador,Estudio,Talleres o cursos,Falta de acceso,Si,Diariamente,Muy importante,Pregrado,Ninguno,Muy alta
1,Susana Alvarez,25-34,Femenino,Computador,Estudio,Ninguno,Ninguna,No,Diariamente,Muy importante,Técnica o tecnólogo,Ninguno,Muy alta
2,Andres Perez,18-24,Masculino,Computador,Trabajo,Ninguno,Ninguna,No,Diariamente,Muy importante,Secundaria,Indígena,Muy alta
3,Mariana Montoya Rios,0 -18,Femenino,Celular,Redes sociales,Autoaprendizaje,Falta de habilidades,No,Diariamente,Importante,Secundaria,Ninguno,Media
4,Sandra Rojas,45-54,Femenino,Celular,Redes sociales,Ninguno,Ninguna,No,Escasamente,Muy importante,Técnica o tecnólogo,Afrocolombiana,Media
5,Julian Mejia,18-24,Masculino,Tablet,Compras en linea,Autoaprendizaje,Ninguna,Si,Escasamente,Muy importante,Postgrado,Ninguno,Muy alta
6,Camilo Vanegas Zapata,25-34,Masculino,Tablet,Compras en linea,Talleres o cursos,Falta de acceso,Tal vez,Semanalmente,Poco importante,Pregrado,Ninguno,Alta
7,Eduardo Agudelo,65+,Masculino,Celular,Trabajo,Talleres o cursos,Poco interés,Tal vez,Semanalmente,Poco importante,Primaria,Campesino,Media


A continuación, se presenta la interpretación de los resultados de la predicción para cada individuo:

In [ ]:
if results_df is not None:
    interpretations = []
    for index, row in results_df.iterrows():
        interpretation = (
            f"Para {row['Nombre']}, {row['Edad']} de edad y género {row['Genero']}, "
            f"que se conecta regularmente con '{row['¿Con qué dispositivos tecnológicos te conectas regularmente?']}' para '{row['¿Mayormente utilizas internet en tu vida diaria para']}', "
            f'ha recibido formación formal sobre el uso de TIC con {row["¿Has recibido formación formal sobre el uso de Tecnologías de la Información y Comunicaciones?"]}, '
            f'considera el uso de las TIC en su vida diaria {row["¿Qué tan importante consideras el uso de Tecnologías de la Información y Comunicaciones en tu vida diaria?"]}, '
            f'con un nivel educativo de {row["¿Cuál es tu nivel educativo?"]}, '
            f"su predicción para la habilidad en el uso de TIC es: **'{row['Nivel_TIC_Predicho']}'**."
        )
        interpretations.append(interpretation)

    for interp in interpretations:
        print(interp)
else:
    print("No se pueden generar interpretaciones debido a un error en la combinación de datos.")

Para Juan Lopez, 18-24 de edad y género Masculino, que se conecta regularmente con 'Computador' para 'Estudio', ha recibido formación formal sobre el uso de TIC con Talleres o cursos, considera el uso de las TIC en su vida diaria Muy importante, con un nivel educativo de Pregrado, su predicción para la habilidad en el uso de TIC es: **'Muy alta'**.
Para Susana Alvarez, 25-34 de edad y género Femenino, que se conecta regularmente con 'Computador' para 'Estudio', ha recibido formación formal sobre el uso de TIC con Ninguno, considera el uso de las TIC en su vida diaria Muy importante, con un nivel educativo de Técnica o tecnólogo, su predicción para la habilidad en el uso de TIC es: **'Muy alta'**.
Para Andres Perez, 18-24 de edad y género Masculino, que se conecta regularmente con 'Computador' para 'Trabajo', ha recibido formación formal sobre el uso de TIC con Ninguno, considera el uso de las TIC en su vida diaria Muy importante, con un nivel educativo de Secundaria, su predicción para